# LightGent LLM server — Colab **TPU v5e-1** edition (EXPERIMENTAL)

Same idea as `colab_llm_server.ipynb` but on Colab's TPU runtime instead of a GPU: [vllm-tpu](https://pypi.org/project/vllm-tpu/) serves an open-source model through the OpenAI-compatible API, exposed via a Cloudflare quick tunnel.

**Setup:** Runtime → Change runtime type → **TPU (v5e-1)**, then Run All.

**Why experimental:** vllm-tpu is built for Cloud TPU VMs; Colab's TPU runtime is close but not identical, and there are few public reports of it working in Colab. If cell 4 or 5 fails, fall back to the GPU notebook — it's the proven path.

**TPU facts that shape this notebook:**
- v5e-1 = one chip, ~16 GB HBM, ~197 bf16 TFLOPs (≈3× a T4) — so the same 4B model, potentially faster.
- No AWQ/GPTQ on TPU — we serve an unquantized bf16 model sized to fit 16 GB.
- First startup XLA-compiles the model: expect an extra 10–20 min before it's ready.

In [ ]:
# ── 1. Config ──────────────────────────────────────────────────────────
API_KEY = "lightgent-change-me"       # anyone with the tunnel URL + this key can use your TPU
MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # bf16 ~8 GB — fits v5e-1's 16 GB HBM; ungated on HF
MAX_MODEL_LEN = 16384                 # lower to 8192 if you hit HBM OOM

In [ ]:
# ── 2. Confirm a TPU is attached ───────────────────────────────────────
import os, sys
print("python:", sys.version.split()[0])
try:
    import jax
    devs = jax.devices()
    print("devices:", devs)
    assert any("TPU" in str(d).upper() for d in devs), "no TPU"
    print("TPU OK")
except Exception as e:
    raise SystemExit(f"No TPU visible ({e}) — set Runtime > Change runtime type > TPU v5e-1")

In [ ]:
# ── 3. Install vllm-tpu + cloudflared (~5-10 min) ──────────────────────
# vllm-tpu pulls its own jax/torch pins and may clash with Colab's
# preinstalled ones. If a later cell fails with an import/version error:
# Runtime > Restart session, then re-run from cell 4 (skip this cell).
!pip install -q -U vllm-tpu openai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("installed")

In [ ]:
# ── 4. Launch vLLM on the TPU ──────────────────────────────────────────
# First run = model download + XLA compilation. Budget 15-30 min total.
import subprocess, time, requests

cmd = [
    "vllm", "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--tensor-parallel-size", "1",
    "--download-dir", "/tmp/models",
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    "--api-key", API_KEY,
]
print(" ".join(cmd))
vllm_log = open("/content/vllm.log", "w")
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

for i in range(270):  # up to 45 min
    if vllm_proc.poll() is not None:
        print(open("/content/vllm.log").read()[-6000:])
        raise SystemExit(
            "vLLM crashed — log above. Common fixes: restart session and rerun "
            "from cell 4; lower MAX_MODEL_LEN to 8192; if the tool-call flags "
            "are rejected remove them (the agent falls back to tag mode); "
            "if it still won't boot, use the GPU notebook instead."
        )
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models",
                         headers={"Authorization": f"Bearer {API_KEY}"}, timeout=3)
        if r.ok:
            print("vLLM ready on TPU:", r.json()["data"][0]["id"])
            break
    except Exception:
        pass
    if i % 6 == 0:
        print(f"waiting for vLLM (download + XLA compile)... ({i*10}s)")
    time.sleep(10)
else:
    raise SystemExit("timed out waiting for vLLM")

In [ ]:
# ── 5. Open the Cloudflare tunnel and print your .env values ───────────
import re, subprocess, time

cf_log = open("/content/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=cf_log, stderr=subprocess.STDOUT,
)
TUNNEL_URL = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        TUNNEL_URL = m.group(0)
        break
assert TUNNEL_URL, "tunnel URL not found — check /content/cloudflared.log"

print("Put these in your lightgent .env (URL rotates on every notebook restart):\n")
print(f"LLM_BASE_URL={TUNNEL_URL}/v1")
print(f"LLM_API_KEY={API_KEY}")
print(f"LLM_MODEL={MODEL}")

In [ ]:
# ── 6. Smoke test through the public tunnel ────────────────────────────
from openai import OpenAI

client = OpenAI(base_url=f"{TUNNEL_URL}/v1", api_key=API_KEY)
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    max_tokens=10,
)
print("tunnel works →", r.choices[0].message.content)

# Tool-calling smoke test (the LightGent agent depends on this — if it fails
# with an error the agent still works via its <tool_call> tag fallback)
try:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "What is the weather in Lagos? Use the tool."}],
        tools=[{"type": "function", "function": {
            "name": "get_weather",
            "description": "Get weather for a city",
            "parameters": {"type": "object",
                            "properties": {"city": {"type": "string"}},
                            "required": ["city"]}}}],
        tool_choice="auto", max_tokens=100,
    )
    tc = r.choices[0].message.tool_calls
    print("native tool calls:", "WORK" if tc else "not used", "→", tc)
except Exception as e:
    print("native tool calls FAILED (agent will use tag-mode fallback):", e)

In [ ]:
# ── 7. (Optional) Keep-alive / monitor — leave this running ────────────
import time

while True:
    tail = open("/content/vllm.log").readlines()[-1:]
    print(time.strftime("%H:%M:%S"), "| tunnel", TUNNEL_URL, "|", (tail[0].strip() if tail else ""))
    time.sleep(300)